In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
import pickle as pkl
from joblib import Parallel, delayed
import joblib
import glob

# Add scripts directory to path
sys.path.append(os.path.abspath("../../scripts"))

# Import modules
import data_pipeline
import xgb_scripts
import plots
from evaluate import evaluate_model, save_results, save_residual_analysis, save_fft_analysis

In [4]:
all_csv_files = glob.glob("../../data/04-03-24/*.csv")
for file_path in all_csv_files:
      df_channel = pd.read_csv(file_path)
      filtered = df_channel[(df_channel['Ns'].isin([1, 6])) & 
                                     (df_channel['freq/Hz'] >= 0.2) & 
                                     (df_channel['freq/Hz'] <= 20000)]
      min_capacity = filtered['Capacity/mA.h'].min()
      max_capacity = filtered['Capacity/mA.h'].max()
      print(f"{os.path.basename(file_path)}: min capacity = {min_capacity:.3f}, max capacity = {max_capacity:.3f}")

A8.csv: min capacity = 0.002, max capacity = 4060.000
A2.csv: min capacity = 0.459, max capacity = 4100.000
A3.csv: min capacity = 5.050, max capacity = 3830.000
A1.csv: min capacity = 0.008, max capacity = 4070.000
A4.csv: min capacity = 0.196, max capacity = 4040.000
A5.csv: min capacity = 6.740, max capacity = 3870.000
A7.csv: min capacity = 0.002, max capacity = 4060.000
A6.csv: min capacity = 0.617, max capacity = 4070.000


In [5]:
def cycle_capacity(df):
    # keep only discharge rows
    d = df[df['I/mA'] < 0].copy()  # or df[df['mode'].eq('Discharge')]
    # one record per (cell, cycle)
    cap = (d.groupby(['cycle number'], as_index=False)
             .agg(discharge_capacity=('Capacity/mA.h','max')))
    return cap


In [6]:
import os, numpy as np, pandas as pd
from collections import Counter, defaultdict

all_bins = np.linspace(3300, 4050, 12)   # adjust if needed
global_hist = Counter()
per_cell_low_share = defaultdict(lambda: [0,0])  # [low_count, total]

def per_cycle_capacity(df):
    d = df[df['I/mA'] < 0]  # discharge only (or df['mode'].eq('Discharge'))
    cap = (d.groupby('cycle number')['Capacity/mA.h']
             .max()                            # end-of-discharge capacity proxy
             .dropna())
    return cap  # pd.Series indexed by cycle number

LOW_THRESH = 3533.0  # from your earlier bins

for fp in all_csv_files:
    cell = os.path.splitext(os.path.basename(fp))[0]  # e.g., "A1"
    df = pd.read_csv(fp)
    df = df[(df['Ns'].isin([1,6])) &
            (df['freq/Hz'].between(0.2, 20000))]

    cap_series = per_cycle_capacity(df)  # one value per cycle
    # update global histogram
    hist, _ = np.histogram(cap_series.values, bins=all_bins)
    global_hist.update({i:int(hist[i]) for i in range(len(hist))})

    # update low share per cell
    low = (cap_series.values < LOW_THRESH).sum()
    tot = cap_series.size
    per_cell_low_share[cell][0] += low
    per_cell_low_share[cell][1] += tot

# Pretty print results
bin_table = pd.DataFrame({
    "bin_left": all_bins[:-1],
    "bin_right": all_bins[1:],
    "count": [global_hist[i] for i in range(len(all_bins)-1)],
})
print(bin_table)

low_share_table = pd.DataFrame({
    "cell_id": list(per_cell_low_share.keys()),
    "low_share": [lo/t if t else 0.0 for lo,t in per_cell_low_share.values()],
    "n_cycles": [t for _,t in per_cell_low_share.values()],
}).sort_values("low_share", ascending=False)
print(low_share_table)


       bin_left    bin_right  count
0   3300.000000  3368.181818    125
1   3368.181818  3436.363636    133
2   3436.363636  3504.545455    150
3   3504.545455  3572.727273    228
4   3572.727273  3640.909091    227
5   3640.909091  3709.090909    158
6   3709.090909  3777.272727    147
7   3777.272727  3845.454545     90
8   3845.454545  3913.636364     45
9   3913.636364  3981.818182     24
10  3981.818182  4050.000000     22
  cell_id  low_share  n_cycles
5      A5   0.785953       299
6      A7   0.784053       301
4      A4   0.703704       270
2      A3   0.679104       268
3      A1   0.626866       268
1      A2   0.611940       268
0      A8   0.548507       268
7      A6   0.052434       267


In [7]:
rows = []
for fp in all_csv_files:
    cell = os.path.splitext(os.path.basename(fp))[0]
    df = pd.read_csv(fp)
    df = df[(df['Ns'].isin([1,6])) &
            (df['freq/Hz'].between(0.2, 20000))]
    cap = per_cycle_capacity(df)
    rows.append(pd.DataFrame({
        "cell_id": cell,
        "cycle": cap.index.values,
        "capacity": cap.values
    }))

# This is the ONLY concat — tiny and safe (one row per cycle)
index_df = pd.concat(rows, ignore_index=True)


In [9]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import numpy as np

n_splits = 5

# 1) Pick a number of bins that won’t create tiny classes
for q in range(10, 4, -1):  # try 10 down to 5 bins
    y_bins = pd.qcut(index_df['capacity'], q=q, duplicates='drop')
    if y_bins.value_counts().min() >= n_splits:
        break

# 2) Convert to valid class labels
y_strat = y_bins.cat.codes.to_numpy()      # ints 0..q-1
groups  = index_df['cell_id'].to_numpy()

# (optional) sanity: every class appears in >= n_splits groups?
ct = pd.crosstab(index_df['cell_id'], y_strat)
# print((ct>0).sum(axis=0))

# 3) Split
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
tr_idx, te_idx = next(sgkf.split(index_df, y_strat, groups))

train_index_meta = index_df.iloc[tr_idx]
test_index_meta  = index_df.iloc[te_idx]
